# 11. Обработка выбросов

Рабочий ноутбук к главе книги `../chapters/11-outliers.md`: здесь только задания и код, теория и выводы — в главе. Источник — глава 17 пособия.

Графики для главы сохраняй через `save_fig(fig, "имя")` → `../chapters/img/11-имя.png`.

In [ ]:
import logging
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etna.datasets import TSDataset, generate_periodic_df
from etna.transforms import (
    MedianOutliersTransform,
    DensityOutliersTransform,
    PredictionIntervalOutliersTransform,
    IQROutlierTransform,
    TimeSeriesImputerTransform,
)
from etna.models import SeasonalMovingAverageModel, ProphetModel
from etna.pipeline import Pipeline
from etna.metrics import SMAPE
from etna.analysis import (
    get_anomalies_median,
    get_anomalies_density,
    get_anomalies_prediction_interval,
    get_anomalies_iqr,
    plot_anomalies,
)

logging.getLogger("cmdstanpy").disabled = True
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = "../local/data"
IMG_DIR = "../chapters/img"  # графики для главы книги
HORIZON = 14
RANDOM_SEED = 42


def save_fig(fig, name: str) -> None:
    """Сохранить график для главы книги: ../chapters/img/11-<name>.png."""
    os.makedirs(IMG_DIR, exist_ok=True)
    fig.savefig(f"{IMG_DIR}/11-{name}.png", dpi=150, bbox_inches="tight")

## Задание 1. Синтетика: найти вставленные выбросы

Ряд, в котором мы знаем, где выбросы.

### Шаг 1.1. Ряд с выбросами

Синтетический ряд с недельной сезонностью и 5 вставленными выбросами (вверх и вниз), позиции сохрани.

### Шаг 1.2. Методы поиска

`get_anomalies_median`, `get_anomalies_density`, `get_anomalies_iqr`, `get_anomalies_prediction_interval` — сколько выбросов нашёл каждый и сколько ложных; `plot_anomalies`.

### Шаг 1.3. Чувствительность

Как меняется результат от параметров (`window_size`, `distance_coef`, `n_neighbors`).

## Задание 2. Выбросы в реальных юнитах

2 юнита `10+` с 2026-04-01.

### Шаг 2.1. Загрузка

Прочитай `facts-by-date.csv` и `units.csv` из `DATA_DIR`, собери `TSDataset` (дневная частота, `segment` = `MaskedUnitId`). Период — с 2026-04-01, 2 юнита `10+`.

### Шаг 2.2. Что находят методы

Аномалии разными методами: какие дни попали и что это за дни.

## Задание 3. Выбросы и прогноз

Стоит ли убирать выбросы перед прогнозом.

### Шаг 3.1. Конвейер с обработкой

`Pipeline(model, transforms=[<OutliersTransform>, TimeSeriesImputerTransform(...)])` — выбросы превращаются в пропуски и заполняются.

### Шаг 3.2. Backtest

Backtest с обработкой выбросов против без неё: SMAPE по фолдам и по юнитам.